# 1. Comparing data

CS 301 · Unit II essentials practice

**Why we do this.** Comparisons help us find patterns in data. Their meaning depends on what the entries represent and how we prepare them.

**By the end, you should be able to:**

- Compute and interpret observation-by-observation and feature-by-feature products.
- Choose and implement cosine similarity or correlation.
- Explain how missing ratings and centring affect a comparison.

Estimated working time: **40–50 minutes**, including written responses.

## Working through this notebook

Save your own copy. Run the setup, then work in order. Before each exercise, write a short prediction. Replace `...` in the exercise cells, run the checks, and explain the result in your own words. A passing check verifies the calculation; the written response shows how you interpret it.

Each notebook runs independently. No results from another notebook are needed. The examples use modest arrays; a GPU is optional in Notebook 3.

**Name:** …


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=4, suppress=True)

def ready(*values):
    assert all(v is not Ellipsis and v is not None for v in values), "Replace every ... in the exercise before running its check."

def heatmap(A, title, labels=None, limits=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    scale = {} if limits is None else {"vmin": limits[0], "vmax": limits[1]}
    im = ax.imshow(A, cmap="coolwarm", aspect="auto", **scale)
    ax.set_title(title)
    if labels is not None:
        ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
        ax.set_yticks(range(len(labels)), labels)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()


## 1. What does each product count?

The rows of `B` are people; the columns are movies. A 1 means a rating was recorded, and a 0 means none was recorded. Entries are integers, so matrix multiplication counts shared ratings.

For a matrix with n rows and m columns, `B.T @ B` compares columns and `B @ B.T` compares rows. Each output entry is a dot product.


In [ ]:
B = np.array([[1, 1, 0], [1, 0, 1], [0, 1, 1], [1, 1, 1]])
display(pd.DataFrame(B, index=["P1", "P2", "P3", "P4"], columns=["A", "B", "C"]))


**Predict before running.** Give the shape of each comparison matrix. What will entry `[0, 1]` mean in each one? What will a diagonal entry count?

*Write your prediction here.*


In [ ]:
movie_overlap = ...
person_overlap = ...


In [ ]:
ready(movie_overlap, person_overlap)
assert movie_overlap.shape == (3, 3), "The movie comparison needs one row and column per movie."
assert person_overlap.shape == (4, 4), "The person comparison needs one row and column per person."
for i in range(3):
    for j in range(3):
        assert movie_overlap[i, j] == np.count_nonzero((B[:, i] == 1) & (B[:, j] == 1)), "Check the shared-rater count."
for i in range(4):
    for j in range(4):
        assert person_overlap[i, j] == np.count_nonzero((B[i] == 1) & (B[j] == 1)), "Check the shared-movie count."
heatmap(movie_overlap, "People who rated both movies", ["A", "B", "C"])
print("Check passed.")


**Explain.** Explain the two `[0, 1]` entries using the original table. Would a large shared-rater count establish that two movies were liked?

*Write your response here.*


## 2. Compare vectors using cosine similarity

Raw dot products depend on vector lengths. Cosine similarity compares direction by dividing the dot product by both lengths. Here each **row** is a vector. All rows used in this exercise have nonzero length.

Implement `row_cosines(X)` without loops. Divide each row by its length using `np.linalg.norm(..., axis=1, keepdims=True)`, then compute all row comparisons.


**Predict before running.** The rows below include two vectors pointing in the same direction with different lengths. Which pair should have cosine similarity 1? Can scaling one row change its raw dot products?

*Write your prediction here.*


In [ ]:
def row_cosines(X):
    U = ...
    return ...

V = np.array([[1., 0.], [2., 0.], [1., 1.], [0., 3.]])
cosines = row_cosines(V)


In [ ]:
ready(cosines)
assert cosines.shape == (4, 4), "Compare the four rows."
assert np.allclose(np.diag(cosines), 1), "A nonzero vector has cosine 1 with itself."
assert np.isclose(cosines[0, 1], 1), "Parallel vectors should have cosine 1."
assert np.isclose(cosines[0, 3], 0), "Perpendicular vectors should have cosine 0."
assert np.isclose(cosines[0, 2], 1 / np.sqrt(2)), "Check normalization."
assert np.allclose(row_cosines(V * np.array([[3], [1], [5], [2]])), cosines), "Positive row scaling should not change cosines."
heatmap(cosines, "Cosine similarity between rows")
print("Check passed.")


**Explain.** Why do row lengths matter to a raw dot product? What goes wrong when you try to normalize a row of zeros?

*Write your response here.*


## 3. Prepare real ratings

We use the same [MovieLens data](https://ikoutis.github.io/cs301-labs/unit-2/movies/SOURCES.md) as Meeting 14, under its [dataset license](https://ikoutis.github.io/cs301-labs/unit-2/movies/README.txt). To keep comparisons small, this notebook uses the 80 most-rated movies. This selection favours popular movies; conclusions apply to this subset.

`R` has one row per person and one column per movie. `NaN` marks a missing rating. Build two versions: missing ratings replaced by zero, and ratings centred by each person's mean before filling missing entries with zero. Use `np.nanmean`, which ignores missing entries.


In [ ]:
BASE = "https://ikoutis.github.io/cs301-labs/unit-2/movies/"
ratings = pd.read_csv(BASE + "ratings.csv")
movies = pd.read_csv(BASE + "movies.csv").set_index("movieId")
chosen = ratings.groupby("movieId").size().nlargest(80).index
table = ratings[ratings.movieId.isin(chosen)].pivot(index="userId", columns="movieId", values="rating")
table = table.dropna(how="all")
R = table.to_numpy()
titles = movies.loc[table.columns, "title"].to_numpy()
print("R shape:", R.shape, "Observed fraction:", np.isfinite(R).mean())
display(pd.DataFrame(R[:5, :6], columns=titles[:6]))


**Predict before running.** What does zero represent before centring? What does zero represent after subtracting a person’s mean rating?

*Write your prediction here.*


In [ ]:
raw = ...
person_mean = ...                 # keep a column shape for broadcasting
centred = ...


In [ ]:
ready(raw, person_mean, centred)
assert person_mean.shape == (len(R), 1), "Retain one mean per person as a column."
observed = np.isfinite(R)
assert np.allclose(raw[observed], R[observed]), "Keep the observed ratings unchanged in raw."
assert np.all(raw[~observed] == 0) and np.all(centred[~observed] == 0), "Fill missing entries with zero."
assert np.allclose(centred.sum(axis=1), 0), "Observed deviations from each person's mean should sum to zero."
expected_mean = np.array([row[np.isfinite(row)].mean() for row in R])[:, None]
assert np.allclose(person_mean, expected_mean), "Compute means over observed ratings only."
assert np.allclose(centred[observed], (R - person_mean)[observed]), "Subtract each person's mean from their observed ratings."
print("Check passed.")


**Explain.** If someone rates two movies below their usual level, what signs do the two centred ratings have? How do they contribute to a similarity dot product?

*Write your response here.*


## 4. Use the comparison to investigate a movie

Use `row_cosines` from Exercise 2 to compare **movie columns** of `raw` and `centred`. Decide which array orientation the function needs. The supplied display code lists neighbours of a movie you choose.


In [ ]:
S_raw = ...
S_centred = ...


In [ ]:
ready(S_raw, S_centred)
for M, S in [(raw, S_raw), (centred, S_centred)]:
    assert S.shape == (len(titles), len(titles)), "Compare movies, not people."
    lengths = np.linalg.norm(M, axis=0)
    assert np.all(lengths > 0), "This comparison requires nonzero movie vectors."
    assert np.allclose(S, (M.T @ M) / np.outer(lengths, lengths)), "Check column normalization."
print("Check passed.")
display(pd.DataFrame({"index": np.arange(len(titles)), "title": titles}).head(20))


In [ ]:
MOVIE = 0  # Choose an index from 0 to len(titles) - 1, then rerun this cell.
assert 0 <= MOVIE < len(titles)
print("Selected movie:", titles[MOVIE])
for label, S in [("Raw ratings", S_raw), ("Centred ratings", S_centred)]:
    scores = S[MOVIE].copy()
    scores[MOVIE] = -np.inf
    near = np.argsort(-scores)[:5]
    print(label)
    display(pd.DataFrame({"movie": titles[near], "similarity": scores[near]}))


**Explain.** Choose a movie, compare its two lists, and report one difference with the actual similarity values. What changed in the question being asked? Give one reason these lists cannot establish what all viewers like.

*Write your response here.*


## 5. Compare features by correlation

Now each row describes a movie. `F` contains rating count, mean rating, and rating standard deviation. Standardize **each column**, using the population standard deviation (`ddof=0`, NumPy's default). Then compute `Z.T @ Z / n`, where n is the number of movies.

All three columns have nonzero standard deviation. Constant columns would need to be removed or handled separately.


In [ ]:
feature_names = ["rating count", "mean rating", "rating spread"]
F = np.column_stack([np.isfinite(R).sum(axis=0), np.nanmean(R, axis=0), np.nanstd(R, axis=0)])


**Predict before running.** What shape will the correlation matrix have? What should its diagonal contain? Predict the sign of the mean-rating/spread relationship and give a reason.

*Write your prediction here.*


In [ ]:
Z = ...
corr = ...


In [ ]:
ready(Z, corr)
assert Z.shape == F.shape
assert np.allclose(Z.mean(axis=0), 0) and np.allclose(Z.std(axis=0), 1), "Standardize features column by column."
assert np.allclose(corr, np.corrcoef(F, rowvar=False)), "Check the product and divisor."
display(pd.DataFrame(corr, index=feature_names, columns=feature_names))
heatmap(corr, "Correlation of movie features", feature_names, limits=(-1,1))
plt.scatter(F[:, 1], F[:, 2], s=18)
plt.xlabel("Mean rating"); plt.ylabel("Rating standard deviation"); plt.show()
print("Check passed.")


**Explain.** Use one off-diagonal value and the scatter plot to describe a relationship. Explain how the five-star ceiling constrains the spread when the mean is near five. Would multiplying rating counts by 100 change their correlations? Explain.

*Write your response here.*


## Final check

Restart the runtime and run your completed notebook from the top. Check that every exercise passes, your figures appear, and every prediction and explanation is filled in. Save the notebook with its outputs.

**Source activities:** Meeting 14 movie-rating activities, Meeting 10 standardization and word-vector comparisons, and vectorization homework. This practice version reorganizes those activities in the order of the Meeting 15 essentials deck.
